# Cost and accuracy of long-term memory in DMAS — Statistical analysis

Reproduces the analysis in *Wolff & Bennati (2026)* by merging every `{prefix}{memory}_{mode}.csv` file under `experiments/results/` into a single DataFrame. Each `(framework, mode)` writes to one file; rows for multiple convs share the file and are distinguished by the `conversation_index` column. Each section maps to a table or figure in the paper.

1. Financial cost — tokens and USD per memory × regime × phase
2. Computational cost — CPU / RAM / disk / network split by cloud vs. edge × phase
2b. Responder context length per question
3. Temporal cost — wall time per memory × regime × phase
4. Response distribution — correct / wrong / IDK
5. Accuracy with 95 % Wilson confidence intervals
6. Two-proportion z-tests
6b. LLM-as-judge agreement (mean correct-vote ratio across `LLM_AS_JUDGE_SEED` calls)
7. Total cost of ownership using AWS Fargate pricing
8. Statistical Pareto-efficiency decision
9. Accuracy split by question category (per judge — keeps cat 5 visible)
9b. Accuracy table by LoCoMo category

**Methodology notes**
- LoCoMo j-score: accuracy is computed on categories 1–4 only; category 5 (adversarial) is excluded by default in §1–§8 (see [zep-papers issue #5](https://github.com/getzep/zep-papers/issues/5)).
- Each question is asked **once**. The LLM judge runs `LLM_AS_JUDGE_SEED` independent times (default 3) and the per-row `judge` column is the majority vote (>50% CORRECT ⇒ CORRECT). `judge_correct_votes / judge_n` and `judge_labels` preserve the raw ballots so §6b can report inter-judge agreement.
- Identical responder system prompt across mem0 and Graphiti — the prompt is not forked per backend (mem0/Zep dispute, May 2025; we side with the "uniform prompt across baselines" position).
- Both memory backends use loading and retrieval logic verbatim from their authors' upstream evaluation harnesses (`mem0ai/memory-benchmarks`, `getzep/zep-papers`).

In [ ]:
from __future__ import annotations
import math
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path('results')
JUDGE = 'judge_verdict'  # majority-vote column written by benchmark/app/results.py

# LoCoMo j-score protocol scores categories 1-4 only; category 5 (adversarial)
# is excluded from accuracy aggregations (per zep-papers issue #5).
SCORED_CATEGORIES = {1, 2, 3, 4}
LOCOMO_CATEGORY_LABEL = {1: 'single-hop', 2: 'multi-hop', 3: 'temporal', 4: 'open-domain', 5: 'adversarial'}

# experiment_name = `{prefix}{memory}_conv{N}_{mode}`. Strip the
# `_conv{N}_{mode}` suffix to recover `{prefix}{memory}` — the readable
# session label used as the leftmost groupby key in every analysis below.
# Conversations and re-runs that share this label aggregate together;
# different `name_prefix` values keep different runs separate. The
# trailing run_id hash from session_id is intentionally NOT in the
# label so tables/graphs stay readable.
_EXP_RE = re.compile(r'_conv\d+_(?:un)?constrained$')

def _experiment_label(experiment_name: str) -> str:
    return _EXP_RE.sub('', str(experiment_name)) if pd.notna(experiment_name) else ''

def load_results(directory: Path = RESULTS_DIR) -> pd.DataFrame:
    """Merge every CSV under `directory` into a single DataFrame.

    The bench writes one file per `(name_prefix, memory, conv, mode)` slug
    (e.g. `test_mem0_conv0_unconstrained.csv`). All files share the same
    column schema written by `benchmark/app/results.py:COLUMNS`, so a
    plain concat produces the union view the analyses below expect.
    A legacy `results.csv` is also picked up if present.
    """
    d = Path(directory)
    if not d.exists():
        return pd.DataFrame()
    paths = sorted(p for p in d.glob('*.csv') if p.stat().st_size > 0)
    if not paths:
        return pd.DataFrame()
    frames = []
    for p in paths:
        # `question` carries free text on ask rows and an int counter on
        # load rows. Force string dtype so per-column inference doesn't
        # flip type depending on whether load or ask rows landed first.
        frames.append(pd.read_csv(p, dtype={'question': str}))
    return pd.concat(frames, ignore_index=True)

def is_unknown(answer) -> bool:
    # The responder is instructed to reply verbatim with
    # "I don't know based on the given context." when the retrieved
    # context is insufficient (see dmas/responder/app/responder_service.py).
    # We bucket those rows as UNKNOWN so they don't count as WRONG.
    return isinstance(answer, str) and "don't know based on the given context" in answer.lower()

def wilson_ci(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    margin = z * math.sqrt((p * (1 - p) + z * z / (4 * n)) / n) / denom
    return max(0.0, center - margin), min(1.0, center + margin)

def two_proportion_ztest(k1: int, n1: int, k2: int, n2: int) -> tuple[float, float]:
    if n1 == 0 or n2 == 0:
        return (float('nan'), float('nan'))
    p1, p2 = k1 / n1, k2 / n2
    p_pool = (k1 + k2) / (n1 + n2)
    se = math.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    if se == 0:
        return (0.0, 1.0)
    z = (p1 - p2) / se
    p = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))
    return z, p

raw = load_results()
if raw.empty:
    print('no per-experiment CSVs in', RESULTS_DIR, '— run `make experiment` first.')
    df_load = df_ask = df = pd.DataFrame()
else:
    constrained = (raw['toxic_latency'].fillna(0) > 0) | (raw['toxic_jitter'].fillna(0) > 0) | (raw['toxic_bandwidth'].fillna(0) > 0)
    raw['regime'] = constrained.map({True: 'constrained', False: 'unconstrained'})
    raw['unknown'] = raw['answer'].apply(is_unknown)
    raw['experiment'] = raw['experiment_name'].apply(_experiment_label)
    # `question_category` only carries a LoCoMo question type on ask
    # rows. Load rows leave it null on purpose so the mapping doesn't
    # claim a session ordinal is e.g. "single-hop". `question_type`
    # is therefore only meaningful on ask rows.
    raw['question_type'] = raw['question_category'].map(LOCOMO_CATEGORY_LABEL)
    # phase=warmup carries one-time backend init cost (graphiti index
    # build, qdrant collection create) — separated so it doesn't
    # spike row #1 of the load. Excluded from accuracy and load/ask
    # totals; surfaced separately in the warmup view below.
    df_load   = raw[raw['phase'] == 'load'].copy()
    df_ask    = raw[raw['phase'] == 'ask'].copy()
    df_warmup = raw[raw['phase'] == 'warmup'].copy()
    # Default accuracy view excludes cat 5 (adversarial). §9 keeps the
    # per-category breakdown unfiltered so cat-5 behaviour stays visible.
    df = df_ask[df_ask['question_category'].isin(SCORED_CATEGORIES)].copy()
    print(f'rows: total={len(raw)}  load={len(df_load)}  ask={len(df_ask)}  scored(cat1-4)={len(df)}')
    print(f'experiments: {sorted(raw.experiment.dropna().unique())}')
    print(f'memories: {sorted(raw.memory.dropna().unique())}  regimes: {sorted(raw.regime.unique())}')
raw.head()


# ----------------------------------------------------------------------
# Plotting helpers shared by every section.
# `plot_metric_per_memory` draws a side-by-side bar chart with
# `unconstrained` vs `constrained` per memory and prints / plots the
# constrained-minus-unconstrained delta beneath it. `experiment` is
# collapsed across because each prefix carries exactly one memory in
# the typical run.
# ----------------------------------------------------------------------
import numpy as np

REGIME_COLOR = {"unconstrained": "#3b82f6", "constrained": "#ef4444"}

def _agg_by_memory_regime(df_in, value_col: str, agg: str = "sum"):
    """Aggregate df_in[value_col] by (memory, regime)."""
    g = df_in.groupby(["memory", "regime"])[value_col]
    if agg == "sum":
        out = g.sum()
    elif agg == "mean":
        out = g.mean()
    elif agg == "median":
        out = g.median()
    else:
        raise ValueError(agg)
    return out.unstack("regime")  # index=memory, columns=[unconstrained, constrained]

def plot_grouped_bars(pivot, *, title: str, ylabel: str,
                      delta_ylabel: str | None = None,
                      ax_pair=None, log_y: bool = False):
    """Render two stacked subplots:
       (top) one bar per memory per regime,
       (bottom) constrained - unconstrained delta per memory.
    `pivot` must have memory on the index, regime in columns.
    """
    # Make sure both regimes exist as columns; missing ones are zero.
    for r in ("unconstrained", "constrained"):
        if r not in pivot.columns:
            pivot[r] = 0.0
    pivot = pivot[["unconstrained", "constrained"]].fillna(0.0)
    delta = pivot["constrained"] - pivot["unconstrained"]

    if ax_pair is None:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(6, 1.3 * len(pivot) + 4), 6),
                                       gridspec_kw={"height_ratios": [3, 2]},
                                       sharex=True)
    else:
        ax1, ax2 = ax_pair
        fig = ax1.figure

    x = np.arange(len(pivot))
    w = 0.38
    ax1.bar(x - w / 2, pivot["unconstrained"], w,
            label="unconstrained", color=REGIME_COLOR["unconstrained"])
    ax1.bar(x + w / 2, pivot["constrained"], w,
            label="constrained", color=REGIME_COLOR["constrained"])
    ax1.set_ylabel(ylabel)
    ax1.set_title(title)
    if log_y:
        ax1.set_yscale("log")
    ax1.legend(fontsize=8)
    ax1.grid(axis="y", alpha=0.25)

    bar_colors = ["#10b981" if v <= 0 else "#ef4444" for v in delta]
    ax2.bar(x, delta, color=bar_colors)
    ax2.axhline(0, color="black", linewidth=0.6)
    ax2.set_ylabel(delta_ylabel or f"Δ ({ylabel})\nconstrained − unconstrained")
    ax2.set_xticks(x)
    ax2.set_xticklabels(pivot.index, rotation=0, fontsize=9)
    ax2.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    return fig, (ax1, ax2), delta


## 1. Financial cost — LLM tokens & USD per memory × regime × phase

Mirrors **Table 1** with both the loading and Q&A phases. Tokens are split by where the LLM is served:
- `edge_llm_*` — local ollama (qwen). Free in litellm pricing.
- `cloud_llm_*` — OpenAI passthrough.
- `llm_*` — sum.

In [ ]:
if raw.empty:
    print('(empty)')
else:
    fin = (
        raw.groupby(['memory', 'regime', 'phase'])
           .agg(
               edge_tokens=('edge_llm_tokens', 'sum'),
               edge_cost=('edge_llm_cost_usd', 'sum'),
               cloud_tokens=('cloud_llm_tokens', 'sum'),
               cloud_cost=('cloud_llm_cost_usd', 'sum'),
               tokens=('llm_tokens', 'sum'),
               cost_usd=('llm_cost_usd', 'sum'),
               n=('phase', 'size'),
           )
           .reset_index()
    )
    metric_cols = ['edge_tokens', 'edge_cost', 'cloud_tokens', 'cloud_cost', 'tokens', 'cost_usd']
    fin = fin[(fin[metric_cols] != 0).any(axis=1)].reset_index(drop=True)
    keep = ['memory', 'regime', 'phase', 'n'] + [c for c in metric_cols if (fin[c] != 0).any()]
    fin = fin[keep]
    print(fin.to_string(index=False, float_format='{:.4f}'.format))

    # Bar chart per phase, grouped by memory × regime, with a Δ panel
    # underneath each (constrained − unconstrained).
    phases_present = [p for p in ('warmup', 'load', 'ask') if p in fin['phase'].unique()]
    for phase in phases_present:
        sub = (fin[fin['phase'] == phase]
                .pivot_table(index='memory', columns='regime', values='cost_usd', aggfunc='sum')
                .fillna(0.0))
        if sub.values.sum() == 0:
            continue
        plot_grouped_bars(sub,
                          title=f'§1 LLM cost — phase={phase}',
                          ylabel='USD (sum across runs)')
        plt.show()


## 2. Computational cost — CPU / RAM / disk / network split by cloud vs. edge × phase

Mirrors **Table 2**. CPU is summed nanoseconds → minutes; RAM is the average bytes-in-use over each row's wall window; disk and network are summed bytes → MB. Split by phase (load vs ask).

> Edge group contains both `coordinator` (lightweight) and `ollama` (heavy local LLM). During the LOAD phase ollama is idle, so `cpu_edge` reflects coordinator overhead only. During ASK, ollama is the dominant edge consumer. Expect very different magnitudes between the two phases on the edge axis. Network is tx-only with toxiproxy excluded so each transferred byte is counted once at its sender.

In [ ]:
if raw.empty:
    print('(empty)')
else:
    g = raw.groupby(['memory', 'regime', 'phase'])
    comp = pd.DataFrame({
        'cpu_cloud_min':    g['cpu_cloud_ns'].sum() / 1e9 / 60,
        'cpu_edge_min':     g['cpu_edge_ns'].sum()  / 1e9 / 60,
        'ram_cloud_MB_peak': g['ram_cloud_peak_bytes'].sum() / 1e6,
        'ram_edge_MB_peak':  g['ram_edge_peak_bytes'].sum()  / 1e6,
        'disk_cloud_MB':    g['disk_cloud_bytes'].sum() / 1e6,
        'disk_edge_MB':     g['disk_edge_bytes'].sum()  / 1e6,
        'network_cloud_MB': g['network_cloud_bytes'].sum() / 1e6,
        'network_edge_MB':  g['network_edge_bytes'].sum()  / 1e6,
    })
    comp = comp[(comp != 0).any(axis=1)]
    comp = comp.loc[:, (comp != 0).any(axis=0)]
    print(comp.to_string(float_format='{:.2f}'.format))

    # Bar charts: one per resource axis. Aggregate across phases so the
    # picture is "total resource use per memory × regime", with the
    # constrained-minus-unconstrained delta directly below.
    resource_axes = [
        ('cpu_cloud_min',    'cloud CPU (minutes)'),
        ('cpu_edge_min',     'edge CPU (minutes)'),
        ('ram_cloud_MB_peak', 'cloud RAM peak (MB·rows)'),
        ('ram_edge_MB_peak',  'edge RAM peak (MB·rows)'),
        ('disk_cloud_MB',    'cloud disk (MB)'),
        ('disk_edge_MB',     'edge disk (MB)'),
        ('network_cloud_MB', 'cloud network tx (MB)'),
        ('network_edge_MB',  'edge network tx (MB)'),
    ]
    for col, ylabel in resource_axes:
        if col not in comp.columns or (comp[col] == 0).all():
            continue
        pivot = (comp[col].groupby(level=['memory', 'regime']).sum().unstack('regime').fillna(0.0))
        plot_grouped_bars(pivot,
                          title=f'§2 {ylabel} per memory',
                          ylabel=ylabel)
        plt.show()


## 2b. Responder context length per question

Mean / median `responder_context_tokens` per `(memory, regime)` on `phase=ask` rows. This is the `prompt_tokens` of the OpenAI completion that produced the final answer — i.e. the actual context length the answering LLM had to process for each question. Higher = the backend handed the responder more retrieved material to read on every question, raising both per-question latency and cost. Reading paired with §1 (cost) and §5/§9b (accuracy) makes the efficiency trade-off explicit: how many tokens does each backend force the responder to consume per correct answer?

In [ ]:
if df_ask.empty or "responder_context_window_tokens" not in df_ask.columns:
    print("(no responder_context_window_tokens column — re-run the bench to populate it)")
else:
    ctx = df_ask.dropna(subset=["responder_context_window_tokens"]).copy()
    if ctx.empty:
        print("(no rows with responder_context_window_tokens recorded — older runs predate this column)")
    else:
        ctx["responder_context_window_tokens"] = ctx["responder_context_window_tokens"].astype(int)
        ctx_summary = (
            ctx.groupby(["memory", "regime"])["responder_context_window_tokens"]
               .agg(n="count", mean="mean", median="median",
                    p95=lambda s: s.quantile(0.95), max="max")
               .reset_index()
        )
        scored = ctx[ctx["question_category"].isin(SCORED_CATEGORIES)].copy()
        if not scored.empty:
            scored["correct"] = (scored[JUDGE] == "CORRECT").astype(int)
            eff = (
                scored.groupby(["memory", "regime"])
                      .agg(total_tokens=("responder_context_window_tokens", "sum"),
                           correct=("correct", "sum"),
                           asked=("correct", "size"))
                      .reset_index()
            )
            eff["tokens_per_correct"] = eff.apply(
                lambda r: (r["total_tokens"] / r["correct"]) if r["correct"] else float("nan"),
                axis=1,
            )
            eff["tokens_per_question"] = eff["total_tokens"] / eff["asked"]
            ctx_summary = ctx_summary.merge(
                eff[["memory", "regime", "tokens_per_question", "tokens_per_correct"]],
                on=["memory", "regime"], how="left",
            )
        print(ctx_summary.to_string(index=False, float_format='{:,.1f}'.format))

        for col, ylabel in (("mean", "mean prompt tokens"),
                             ("p95", "p95 prompt tokens"),
                             ("tokens_per_correct", "tokens per CORRECT answer")):
            if col not in ctx_summary.columns:
                continue
            pivot = ctx_summary.pivot(index="memory", columns="regime", values=col).fillna(0.0)
            if pivot.values.sum() == 0:
                continue
            plot_grouped_bars(pivot,
                              title=f'§2b {ylabel} per memory',
                              ylabel=ylabel)
            plt.show()


## 3. Temporal cost — wall time per memory × regime × phase

Mirrors **Table 3** with both the loading and Q&A phases.

In [ ]:
if raw.empty:
    print('(empty)')
else:
    # wall_ms = compute_ms + flush_ms.
    #   compute_ms is the user-facing latency a production system would pay.
    #   flush_ms is the bench-side I/O quiescence wait — instrumentation
    #   artifact added so async DB checkpoints land in the row's disk delta.
    # Report the AVERAGE per row (per message during load, per question
    # during ask), not the sum — the totals scaled with dataset size and
    # weren't comparable across phases.
    tcost = (
        raw.groupby(['memory', 'regime', 'phase'])
           .agg(mean_wall_s=('wall_ms', lambda s: s.mean() / 1000),
                mean_compute_s=('compute_ms', lambda s: s.mean() / 1000),
                mean_flush_s=('flush_ms', lambda s: s.mean() / 1000),
                n=('phase', 'size'))
           .reset_index()
    )
    tcost = tcost[tcost['mean_wall_s'] != 0].reset_index(drop=True)
    print(tcost.to_string(index=False, float_format='{:.2f}'.format))

    # One figure per memory framework — separate charts so the eye can
    # scan one backend at a time and the smaller backends aren\'t crushed
    # by graphiti/cognee scale. Each figure has two panels:
    #   (top) one bar per (regime, phase) with compute_ms (production)
    #         and flush_ms (bench artifact) stacked.
    #   (bottom) Δ wall_s = constrained − unconstrained per phase.
    memories = sorted(tcost['memory'].dropna().unique())
    PHASE_ORDER = ['warmup', 'load', 'ask']
    REGIMES = ['unconstrained', 'constrained']
    for mem in memories:
        sub = tcost[tcost['memory'] == mem]
        if sub.empty:
            continue
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.5, 6),
                                       gridspec_kw={"height_ratios": [3, 2]},
                                       sharex=True)

        # Build (regime, phase) tick order so columns line up across runs.
        rows = []
        for regime in REGIMES:
            for phase in PHASE_ORDER:
                m = sub[(sub['regime'] == regime) & (sub['phase'] == phase)]
                if m.empty:
                    continue
                r = m.iloc[0]
                rows.append({'regime': regime, 'phase': phase,
                             'mean_compute_s': r['mean_compute_s'],
                             'mean_flush_s': r['mean_flush_s'],
                             'n': int(r['n'])})
        if not rows:
            plt.close(fig); continue
        plot_df = pd.DataFrame(rows)
        x = np.arange(len(plot_df))
        labels = [f"{r}\n{p} (n={n})" for r, p, n in zip(plot_df['regime'], plot_df['phase'], plot_df['n'])]
        ax1.bar(x, plot_df['mean_compute_s'], label='compute (production cost)', color='#3b82f6')
        if (plot_df['mean_flush_s'] != 0).any():
            ax1.bar(x, plot_df['mean_flush_s'], bottom=plot_df['mean_compute_s'],
                    label='flush (bench artifact)', color='#ef4444',
                    hatch='///', edgecolor='white')
        ax1.set_xticks(x)
        ax1.set_xticklabels(labels, rotation=0, fontsize=8)
        ax1.set_ylabel('mean seconds per row')
        ax1.set_title(f'§3 Mean wall time per row — {mem}')
        ax1.legend(fontsize=8)
        ax1.grid(axis='y', alpha=0.25)

        # Δ panel: per-phase constrained-minus-unconstrained wall_s.
        delta_rows = []
        for phase in PHASE_ORDER:
            u = sub[(sub['regime'] == 'unconstrained') & (sub['phase'] == phase)]
            c = sub[(sub['regime'] == 'constrained') & (sub['phase'] == phase)]
            if u.empty or c.empty:
                continue
            delta_rows.append({'phase': phase,
                               'delta_s': float(c['mean_wall_s'].iloc[0]) - float(u['mean_wall_s'].iloc[0])})
        if delta_rows:
            ddf = pd.DataFrame(delta_rows)
            xd = np.arange(len(ddf))
            colors = ['#10b981' if v <= 0 else '#ef4444' for v in ddf['delta_s']]
            ax2.bar(xd, ddf['delta_s'], color=colors)
            ax2.axhline(0, color='black', linewidth=0.6)
            ax2.set_xticks(xd)
            ax2.set_xticklabels(ddf['phase'], fontsize=9)
            ax2.set_ylabel('Δ wall_s\n(constrained − unconstrained)')
            ax2.grid(axis='y', alpha=0.25)
        else:
            ax2.set_visible(False)
        fig.tight_layout()
        plt.show()


## 4. Response distribution — correct / wrong / unknown

Mirrors **Table 4** and **Figure 2**. A response is UNKNOWN if the responder said “I don't know based on the given context.” (verbatim, case-insensitive). A response is CORRECT iff the primary judge labels it CORRECT; otherwise WRONG (excluding UNKNOWN).

In [ ]:
def classify(row):
    if row['unknown']:
        return 'UNKNOWN'
    if row[JUDGE] == 'CORRECT':
        return 'CORRECT'
    return 'WRONG'

if df.empty:
    print('(empty)')
else:
    from matplotlib.ticker import PercentFormatter
    df['outcome'] = df.apply(classify, axis=1)
    counts = (
        df.groupby(['memory', 'regime', 'outcome']).size().unstack(fill_value=0)
        .reindex(columns=['CORRECT', 'WRONG', 'UNKNOWN'], fill_value=0)
    )
    counts['n'] = counts.sum(axis=1)
    pct = counts[['CORRECT', 'WRONG', 'UNKNOWN']].div(counts['n'], axis=0) * 100
    pct.columns = ['CORRECT_pct', 'WRONG_pct', 'UNKNOWN_pct']
    table = pd.concat([counts['n'], pct], axis=1)
    nonzero_outcomes = [o for o in ['CORRECT', 'WRONG', 'UNKNOWN'] if (pct[f'{o}_pct'] > 0).any()]
    keep_cols = ['n'] + [f'{o}_pct' for o in nonzero_outcomes]
    print(table[keep_cols].to_string(float_format='{:.1f}'.format))

    # One chart, all memories side by side. Two clusters per memory
    # (unconstrained / constrained), each cluster a stacked
    # CORRECT/WRONG/UNKNOWN bar. The constrained bar is drawn with a
    # red border AND hatch AND a "C" label so it can't be mistaken
    # for the unconstrained one even when both bars hold identical
    # values. The Δ panel beneath shows constrained-minus-unconstrained
    # CORRECT% per memory.
    palette = {'CORRECT': '#3b82f6', 'WRONG': '#ef4444', 'UNKNOWN': '#9ca3af'}
    regimes_present = list(pct.index.get_level_values('regime').unique())
    memories = sorted(pct.index.get_level_values('memory').unique())
    regimes = [r for r in ('unconstrained', 'constrained') if r in regimes_present]
    if not memories or not regimes:
        print('(no memory × regime data)')
    else:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(9, 1.8 * len(memories) + 4), 7),
                                       gridspec_kw={"height_ratios": [3, 2]}, sharex=True)
        x = np.arange(len(memories))
        cluster_w = 0.8
        bar_w = cluster_w / len(regimes)
        edge_color = {'unconstrained': 'black', 'constrained': '#7f1d1d'}
        legend_seen = set()
        for ri, regime in enumerate(regimes):
            offset = (ri - (len(regimes) - 1) / 2) * bar_w
            bottoms = np.zeros(len(memories))
            for outcome in nonzero_outcomes:
                vals = []
                for m in memories:
                    try:
                        vals.append(pct.loc[(m, regime), f'{outcome}_pct'])
                    except KeyError:
                        vals.append(0.0)
                vals = np.array(vals)
                lbl = f'{outcome}'
                show_label = lbl not in legend_seen
                if show_label:
                    legend_seen.add(lbl)
                ax1.bar(x + offset, vals, bar_w * 0.92, bottom=bottoms,
                        color=palette[outcome],
                        label=lbl if show_label else None,
                        edgecolor=edge_color[regime],
                        linewidth=1.6 if regime == 'constrained' else 0.6,
                        hatch='///' if regime == 'constrained' else None)
                bottoms += vals
            # Per-bar regime + n label written ON the bar bottom so it
            # renders even when the bar fills the axis (CORRECT=100%).
            for mi, m in enumerate(memories):
                try:
                    n = int(counts.loc[(m, regime), 'n'])
                except KeyError:
                    n = 0
                tag = 'U' if regime == 'unconstrained' else 'C'
                ax1.text(x[mi] + offset, 4, f'{tag}\nn={n}',
                         ha='center', va='bottom', fontsize=8,
                         color='white' if regime == 'unconstrained' else '#fef2f2',
                         fontweight='bold')

        ax1.set_xticks(x)
        ax1.set_xticklabels(memories, fontsize=10)
        ax1.set_ylabel('% of responses')
        ax1.set_ylim(0, 110)
        ax1.yaxis.set_major_formatter(PercentFormatter(decimals=0))
        ax1.set_title('§4 Response distribution per memory '
                      '(U = unconstrained, C = constrained / hatched + red border)')
        ax1.legend(loc='upper right', fontsize=8)
        ax1.grid(axis='y', alpha=0.25)

        # Δ panel: constrained CORRECT% minus unconstrained CORRECT%.
        if {'unconstrained', 'constrained'}.issubset(set(regimes)):
            delta_correct = []
            for m in memories:
                u = pct.loc[(m, 'unconstrained'), 'CORRECT_pct'] if (m, 'unconstrained') in pct.index else 0.0
                c = pct.loc[(m, 'constrained'), 'CORRECT_pct'] if (m, 'constrained') in pct.index else 0.0
                delta_correct.append(c - u)
            colors = ['#10b981' if v >= 0 else '#ef4444' for v in delta_correct]
            ax2.bar(x, delta_correct, color=colors)
            ax2.axhline(0, color='black', linewidth=0.6)
            ax2.set_ylabel('Δ CORRECT %\n(constrained − unconstrained)')
            ax2.set_xticks(x)
            ax2.set_xticklabels(memories, fontsize=10)
            ax2.grid(axis='y', alpha=0.25)
        else:
            ax2.set_visible(False)

        fig.tight_layout()
        plt.show()


## 5. Accuracy with 95 % Wilson confidence intervals

Mirrors **Table 5**. Closed-form Wilson interval, no scipy/statsmodels dependency.

In [ ]:
if df.empty:
    print('(empty)')
else:
    rows = []
    for (memory, regime), sub in df.groupby(['memory', 'regime']):
        n = len(sub)
        k = int((sub[JUDGE] == 'CORRECT').sum())
        lo, hi = wilson_ci(k, n)
        rows.append({'memory': memory, 'regime': regime,
                     'n': n, 'correct': k,
                     'accuracy': k / n if n else 0.0, 'ci_lo': lo, 'ci_hi': hi})
    tab5 = pd.DataFrame(rows).sort_values(['memory', 'regime']).reset_index(drop=True)
    print(tab5.to_string(index=False, float_format='{:.3f}'.format))

    # Bar chart with Wilson 95% CI error bars; Δ panel beneath shows
    # constrained-minus-unconstrained accuracy gap per memory.
    pivot_acc = tab5.pivot(index='memory', columns='regime', values='accuracy').fillna(0.0)
    pivot_lo  = tab5.pivot(index='memory', columns='regime', values='ci_lo').fillna(0.0)
    pivot_hi  = tab5.pivot(index='memory', columns='regime', values='ci_hi').fillna(0.0)
    for r in ('unconstrained', 'constrained'):
        for p in (pivot_acc, pivot_lo, pivot_hi):
            if r not in p.columns:
                p[r] = 0.0
    pivot_acc = pivot_acc[['unconstrained', 'constrained']]
    pivot_lo  = pivot_lo[['unconstrained', 'constrained']]
    pivot_hi  = pivot_hi[['unconstrained', 'constrained']]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(6, 1.3 * len(pivot_acc) + 4), 6),
                                   gridspec_kw={"height_ratios": [3, 2]}, sharex=True)
    x = np.arange(len(pivot_acc))
    w = 0.38
    for i, regime in enumerate(['unconstrained', 'constrained']):
        offset = (i - 0.5) * w
        heights = pivot_acc[regime].values
        err_low = heights - pivot_lo[regime].values
        err_high = pivot_hi[regime].values - heights
        ax1.bar(x + offset, heights, w, label=regime, color=REGIME_COLOR[regime],
                yerr=[err_low, err_high], capsize=3,
                error_kw={"linewidth": 0.8})
    ax1.set_ylabel('accuracy (j-score, cat 1-4)')
    ax1.set_ylim(0, 1)
    ax1.set_title('§5 Accuracy with Wilson 95% CI')
    ax1.legend(fontsize=8)
    ax1.grid(axis='y', alpha=0.25)

    delta = pivot_acc['constrained'] - pivot_acc['unconstrained']
    bar_colors = ["#ef4444" if v < 0 else "#10b981" for v in delta]
    ax2.bar(x, delta, color=bar_colors)
    ax2.axhline(0, color='black', linewidth=0.6)
    ax2.set_ylabel('Δ accuracy\n(constrained − unconstrained)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(pivot_acc.index, rotation=0, fontsize=9)
    ax2.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    plt.show()


## 6. Two-proportion z-tests

Mirrors **Table 6**. α = 0.05. Closed-form pooled-variance two-sided test.

In [ ]:
if df.empty:
    print('(empty)')
else:
    def kn(memory, regime):
        sub = df[(df['memory'] == memory) & (df['regime'] == regime)]
        return int((sub[JUDGE] == 'CORRECT').sum()), len(sub)

    memories = sorted(df['memory'].dropna().unique())
    regimes_present = [r for r in ('unconstrained', 'constrained') if r in df['regime'].unique()]
    comparisons = []
    # Cross-memory comparisons within a regime (every pair with data).
    for regime in regimes_present:
        for i in range(len(memories)):
            for j in range(i + 1, len(memories)):
                a, b = memories[i], memories[j]
                k1, n1 = kn(a, regime); k2, n2 = kn(b, regime)
                if n1 == 0 or n2 == 0:
                    continue
                z, p = two_proportion_ztest(k1, n1, k2, n2)
                comparisons.append({'comparison': f'{a} vs {b} ({regime})',
                                    'k1': k1, 'n1': n1, 'k2': k2, 'n2': n2,
                                    'z_stat': z, 'p_value': p,
                                    'significant_alpha_0.05': bool(p < 0.05)})
    # Per-memory regime-vs-regime gap.
    if {'unconstrained', 'constrained'}.issubset(regimes_present):
        for m in memories:
            k1, n1 = kn(m, 'unconstrained'); k2, n2 = kn(m, 'constrained')
            if n1 == 0 or n2 == 0:
                continue
            z, p = two_proportion_ztest(k1, n1, k2, n2)
            comparisons.append({'comparison': f'{m}: unconstrained vs constrained',
                                'k1': k1, 'n1': n1, 'k2': k2, 'n2': n2,
                                'z_stat': z, 'p_value': p,
                                'significant_alpha_0.05': bool(p < 0.05)})

    if not comparisons:
        print('(no pairs with data on both sides)')
    else:
        tab6 = pd.DataFrame(comparisons)
        print(tab6.to_string(index=False, float_format='{:.4f}'.format))

        # Bar chart of -log10(p_value) per comparison; bars above the
        # dashed α=0.05 line are statistically significant. Sign of
        # (k1/n1 − k2/n2) is annotated as the bar color.
        signs = ((tab6['k1'] / tab6['n1']) - (tab6['k2'] / tab6['n2']))
        colors = ['#10b981' if s > 0 else '#ef4444' if s < 0 else '#9ca3af' for s in signs]
        neg_log_p = -np.log10(tab6['p_value'].replace(0, np.nan))
        fig, ax = plt.subplots(figsize=(max(6, 0.7 * len(tab6) + 4), 4.5))
        x = np.arange(len(tab6))
        ax.bar(x, neg_log_p.fillna(0), color=colors)
        ax.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8, label='α=0.05')
        ax.set_xticks(x)
        ax.set_xticklabels(tab6['comparison'], rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('-log10(p-value)')
        ax.set_title('§6 Two-proportion z-tests\n(green: first side higher | red: second side higher)')
        ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()


## 6b. LLM-as-judge agreement

With the new protocol there is exactly one ask per question; the variability source is the judge, not the responder. The current results schema persists only the majority `judge_verdict` and the panel size `judge_n` (per-call labels and correct-vote counts are no longer written), so this cell can only report the panel size and the majority-correct rate per `(memory, regime)`. The majority-correct rate equals §5 accuracy — kept here for cross-reference.

In [ ]:
if df.empty:
    print('(empty)')
elif "judge_n" not in df.columns or "judge_verdict" not in df.columns:
    print('(no judge_n / judge_verdict columns — older runs predate the LLM_AS_JUDGE_SEED protocol)')
else:
    panel = df.dropna(subset=["judge_n", "judge_verdict"]).copy()
    panel = panel[panel["judge_n"].astype(float) > 0]
    if panel.empty:
        print('(no rows with judge votes recorded)')
    else:
        summary = (
            panel.groupby(["memory", "regime"])
                 .agg(n_questions=("judge_verdict", "size"),
                      majority_correct=("judge_verdict", lambda s: (s == "CORRECT").mean()),
                      judge_n=("judge_n", "max"))
                 .reset_index()
        )
        summary["majority_correct_pct"] = summary["majority_correct"] * 100
        cols = ["memory", "regime", "judge_n", "n_questions", "majority_correct_pct"]
        print(summary[cols].to_string(index=False, float_format='{:.2f}'.format))

        pivot = summary.pivot(index='memory', columns='regime', values='majority_correct_pct').fillna(0.0)
        plot_grouped_bars(pivot,
                          title='§6b Majority-correct rate (= §5 accuracy, cross-check)',
                          ylabel='majority-correct %',
                          delta_ylabel='Δ majority-correct %\n(constrained − unconstrained)')
        plt.show()


## 7. Total cost of ownership (AWS Fargate, us-east-1)

Mirrors **Section 4.1 / Figure 3**. Pricing:

- Compute: \$0.04048 / vCPU-hour
- RAM: \$0.004445 / GB-hour
- Storage (EBS gp3): \$0.000109 / GB-hour
- Network (egress): \$0.09 / GB

In [ ]:
VCPU_PER_HOUR  = 0.04048
RAM_GB_HOUR    = 0.004445
DISK_GB_HOUR   = 0.000109
NET_PER_GB     = 0.09

if raw.empty:
    print('(empty)')
else:
    # Per-row resource cost across BOTH phases (load + ask).
    work = raw.copy()
    # Use compute_ms (production-equivalent latency) for the GB-hour
    # cost multiplier — flush_ms is bench-introduced quiescence wait that
    # a real customer would never pay.
    compute_h = work['compute_ms'] / 1000 / 3600
    for grp in ('edge', 'cloud'):
        cpu_h  = work[f'cpu_{grp}_ns'] / 1e9 / 3600
        ram_gb = work[f'ram_{grp}_peak_bytes'] / 1e9
        disk_gb = work[f'disk_{grp}_bytes'] / 1e9
        net_gb = work[f'network_{grp}_bytes'] / 1e9
        work[f'cost_cpu_{grp}']  = cpu_h * VCPU_PER_HOUR
        work[f'cost_ram_{grp}']  = ram_gb * compute_h * RAM_GB_HOUR
        work[f'cost_disk_{grp}'] = disk_gb * compute_h * DISK_GB_HOUR
        work[f'cost_net_{grp}']  = net_gb * NET_PER_GB
    work['cost_llm_edge']  = work['edge_llm_cost_usd'].fillna(0)
    work['cost_llm_cloud'] = work['cloud_llm_cost_usd'].fillna(0)
    for kind in ('cpu', 'ram', 'disk', 'net', 'llm'):
        work[f'cost_{kind}'] = work[f'cost_{kind}_edge'] + work[f'cost_{kind}_cloud']

    cost_cols = [f'cost_{k}' for k in ('cpu', 'ram', 'disk', 'net', 'llm')]
    tco_full = work.groupby(['memory', 'regime', 'phase'])[cost_cols].sum()
    tco_full['total'] = tco_full[cost_cols].sum(axis=1)
    tco = tco_full[tco_full['total'] != 0]
    nonzero_cost_cols = [c for c in cost_cols if (tco[c] != 0).any()]
    tco = tco[nonzero_cost_cols + ['total']]
    print('Cost breakdown (USD) per memory × regime × phase, grouped by resource type:')
    print(tco.to_string(float_format='{:.6f}'.format))

    # Flat TCO per (memory, regime) for §8 — kept on `experiment` axis
    # for back-compat with §8 which reads it via tco_flat.xs('experiment').
    # We synthesise an `experiment` level so §8\'s indexing still works
    # whether the user runs prefix-per-backend or single-prefix.
    tco_flat_mem = tco.groupby(level=['memory', 'regime'])['total'].sum().to_frame('total')
    # Re-synthesise the (experiment, memory, regime) shape §8 expects by
    # repeating the row for each experiment label seen in `raw`.
    exp_labels = sorted(raw['experiment'].dropna().unique()) or ['']
    tco_flat = pd.concat({e: tco_flat_mem for e in exp_labels},
                         names=['experiment']).reorder_levels(['experiment', 'memory', 'regime']).sort_index()

    # One figure per memory framework. Each figure has two panels:
    #   (top) per-phase stacked bars by resource type; one cluster per
    #         (regime, phase) so the cost composition is readable.
    #   (bottom) Δ TCO total per phase (constrained − unconstrained).
    PHASE_ORDER = ['warmup', 'load', 'ask']
    REGIMES = ['unconstrained', 'constrained']
    resource_palette = {
        'cost_cpu':  '#3b82f6',
        'cost_ram':  '#10b981',
        'cost_disk': '#f59e0b',
        'cost_net':  '#a855f7',
        'cost_llm':  '#ef4444',
    }
    resource_label = {'cost_cpu': 'cpu', 'cost_ram': 'ram', 'cost_disk': 'disk',
                      'cost_net': 'network', 'cost_llm': 'llm'}
    plot_phases = [p for p in PHASE_ORDER if p in tco.index.get_level_values('phase')]
    if not plot_phases:
        print('(no warmup/load/ask cost data)')
    else:
        memories = sorted(tco.index.get_level_values('memory').unique())
        for mem in memories:
            try:
                sub = tco.xs(mem, level='memory')
            except KeyError:
                continue
            sub = sub[sub.index.get_level_values('phase').isin(plot_phases)]
            if sub.empty:
                continue
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.5, 6),
                                           gridspec_kw={"height_ratios": [3, 2]},
                                           sharex=False)

            # Build (regime, phase) order, drop missing combinations.
            ordered = []
            for regime in REGIMES:
                for phase in plot_phases:
                    if (regime, phase) in sub.index:
                        ordered.append((regime, phase))
            if not ordered:
                plt.close(fig); continue
            x = np.arange(len(ordered))
            labels = [f'{r}\n{p}' for r, p in ordered]
            bottom = np.zeros(len(ordered))
            for col in nonzero_cost_cols:
                vals = np.array([sub.loc[(r, p), col] for r, p in ordered])
                if (vals == 0).all():
                    continue
                ax1.bar(x, vals, bottom=bottom, label=resource_label[col],
                        color=resource_palette[col])
                bottom = bottom + vals
            ax1.set_xticks(x)
            ax1.set_xticklabels(labels, fontsize=8)
            ax1.set_ylabel('USD')
            ax1.set_title(f'§7 TCO per phase — {mem}')
            ax1.legend(fontsize=8)
            ax1.grid(axis='y', alpha=0.25)

            # Δ panel: per-phase total cost delta (constrained − unconstrained).
            delta_rows = []
            for phase in plot_phases:
                if ('unconstrained', phase) in sub.index and ('constrained', phase) in sub.index:
                    u = float(sub.loc[('unconstrained', phase), 'total'])
                    c = float(sub.loc[('constrained', phase), 'total'])
                    delta_rows.append({'phase': phase, 'delta': c - u})
            if delta_rows:
                ddf = pd.DataFrame(delta_rows)
                xd = np.arange(len(ddf))
                colors = ['#10b981' if v <= 0 else '#ef4444' for v in ddf['delta']]
                ax2.bar(xd, ddf['delta'], color=colors)
                ax2.axhline(0, color='black', linewidth=0.6)
                ax2.set_xticks(xd)
                ax2.set_xticklabels(ddf['phase'], fontsize=9)
                ax2.set_ylabel('Δ TCO USD\n(constrained − unconstrained)')
                ax2.grid(axis='y', alpha=0.25)
            else:
                ax2.set_visible(False)
            fig.tight_layout()
            plt.show()


## 8. Statistical Pareto-efficiency decision

Per **Section 4.2**: a system $S_A$ strictly dominates $S_B$ iff
1. $C(S_A) < C(S_B)$, AND
2. The accuracy gap $A(S_B) - A(S_A)$ is *not* statistically significant.

Under that definition the optimization collapses to cost-minimization whenever the z-test in §6 fails to reject the null.

In [ ]:
if df.empty:
    print('(empty)')
else:
    # Per-memory per-regime aggregation across whatever experiment
    # prefixes the run wrote. With one prefix per backend (e.g.
    # `test_cognee_*`, `test_mem0_*`) the previous "experiment+regime"
    # group could never see two memories at once, so the dominance
    # check produced no comparisons. Aggregating to (memory, regime)
    # collapses prefixes back together.
    cost_per_mem = (tco_flat
                    .groupby(level=['memory', 'regime'])['total']
                    .sum()
                    .unstack('regime')
                    .fillna(0.0))
    for r in ('unconstrained', 'constrained'):
        if r not in cost_per_mem.columns:
            cost_per_mem[r] = 0.0
    cost_per_mem = cost_per_mem[['unconstrained', 'constrained']]

    def kn(memory, regime):
        sub = df[(df['memory'] == memory) & (df['regime'] == regime)]
        return int((sub[JUDGE] == 'CORRECT').sum()), len(sub)

    # Pairwise dominance within a regime.
    memories = list(cost_per_mem.index)
    regimes_present = [r for r in ('unconstrained', 'constrained')
                       if (cost_per_mem[r] != 0).any()]
    if len(memories) < 2 or not regimes_present:
        print('Pareto comparison needs ≥2 memories and ≥1 regime with data; skipping.')
    else:
        verdicts = []
        for regime in regimes_present:
            for i in range(len(memories)):
                for j in range(i + 1, len(memories)):
                    a, b = memories[i], memories[j]
                    cost_a = cost_per_mem.loc[a, regime]
                    cost_b = cost_per_mem.loc[b, regime]
                    k_a, n_a = kn(a, regime); k_b, n_b = kn(b, regime)
                    if n_a == 0 or n_b == 0:
                        continue
                    z, p = two_proportion_ztest(k_a, n_a, k_b, n_b)
                    statistically_equivalent = p >= 0.05
                    if statistically_equivalent:
                        if cost_a < cost_b:
                            verdict = f'{a} dominates {b} (cheaper, accuracy gap n.s.)'
                        elif cost_b < cost_a:
                            verdict = f'{b} dominates {a} (cheaper, accuracy gap n.s.)'
                        else:
                            verdict = f'tie (same cost, accuracy gap n.s.)'
                    else:
                        verdict = 'real trade-off (accuracy gap IS significant)'
                    print(f'[{regime}]  TCO {a}=${cost_a:.4f}  TCO {b}=${cost_b:.4f}  '
                          f'z={z:.3f}, p={p:.4f}  ⇒ {verdict}')
                    verdicts.append({'regime': regime, 'pair': f'{a} vs {b}',
                                     'cost_a': cost_a, 'cost_b': cost_b,
                                     'p_value': p, 'verdict': verdict})

        # Bar chart: TCO per memory × regime, with Δ panel underneath.
        plot_grouped_bars(cost_per_mem,
                          title='§8 Total cost of ownership per memory (USD)',
                          ylabel='USD (load+ask, AWS Fargate priced)')
        plt.show()


## 9. Accuracy split by question category × judge

Per-judge view (mem0 / zep / dmas) of accuracy across LoCoMo categories × regime.

In [ ]:
if df_ask.empty:
    print('(empty)')
else:
    out = (
        df_ask.groupby(['memory', 'question_type', 'regime'])[JUDGE]
              .apply(lambda s: (s == 'CORRECT').mean())
              .rename('accuracy').reset_index()
    )
    if out.empty:
        print('(no rows)')
    else:
        pivot = out.pivot_table(index=['memory', 'question_type'],
                                columns='regime', values='accuracy').fillna(0.0)
        for r in ('unconstrained', 'constrained'):
            if r not in pivot.columns:
                pivot[r] = 0.0
        pivot = pivot[['unconstrained', 'constrained']]
        print(pivot.to_string(float_format='{:.3f}'.format))

        # Heatmap-style grouped bar per memory, with Δ panel beneath.
        memories = sorted(out['memory'].dropna().unique())
        for mem in memories:
            try:
                sub = pivot.xs(mem, level='memory')
            except KeyError:
                continue
            if sub.empty:
                continue
            plot_grouped_bars(sub,
                              title=f'§9 Accuracy per question type — {mem}',
                              ylabel='accuracy',
                              delta_ylabel='Δ accuracy\n(constrained − unconstrained)')
            plt.show()


### 9b. Accuracy table by LoCoMo category

Same data as §9 but pivoted for readability: rows = `(memory, regime)`, columns = LoCoMo categories. Each cell shows `k/n (acc%)`. The `overall (1-4)` column reproduces the j-score (cat 1-4 only) and matches §5/§6b.

In [ ]:
if df_ask.empty:
    print('(empty)')
else:
    cat_order = [1, 2, 3, 4, 5]
    cat_cols = [LOCOMO_CATEGORY_LABEL[c] for c in cat_order]

    def fmt(k, n):
        if n == 0:
            return '-'
        return f'{k}/{n} ({100 * k / n:.0f}%)'

    rows = []
    for (memory, regime), sub in df_ask.groupby(['memory', 'regime']):
        row = {'memory': memory, 'regime': regime}
        for c in cat_order:
            cat_sub = sub[sub['question_category'] == c]
            n = len(cat_sub)
            k = int((cat_sub[JUDGE] == 'CORRECT').sum())
            row[LOCOMO_CATEGORY_LABEL[c]] = fmt(k, n)
        scored = sub[sub['question_category'].isin(SCORED_CATEGORIES)]
        n_all = len(scored)
        k_all = int((scored[JUDGE] == 'CORRECT').sum())
        row['overall (1-4)'] = fmt(k_all, n_all)
        rows.append(row)
    cat_table = pd.DataFrame(rows).set_index(['memory', 'regime'])[cat_cols + ['overall (1-4)']]
    print(cat_table.to_string())

    # Numeric grouped bar: overall (1-4) accuracy per memory × regime.
    overall = (df_ask[df_ask['question_category'].isin(SCORED_CATEGORIES)]
                .groupby(['memory', 'regime'])[JUDGE]
                .apply(lambda s: (s == 'CORRECT').mean())
                .unstack('regime').fillna(0.0))
    for r in ('unconstrained', 'constrained'):
        if r not in overall.columns:
            overall[r] = 0.0
    overall = overall[['unconstrained', 'constrained']]
    plot_grouped_bars(overall,
                      title='§9b Overall (cat 1-4) accuracy per memory',
                      ylabel='j-score',
                      delta_ylabel='Δ j-score\n(constrained − unconstrained)')
    plt.show()


## Appendix — column glossary

Per-experiment CSV schema (one file per `(prefix, memory, conv, mode)` slug). Three phases:
- `phase=warmup` — one row per `(memory, conv, mode)` leg, written immediately after the pre-leg reset. Captures one-time backend init (graphiti `build_indices_and_constraints`, qdrant collection creation) so it doesn't get folded into row #1 of the load. `question`, `seed`, `category`, `judge*` are null.
- `phase=load` — one row per memory persisted (loaded). `seed` holds the **session number** (1..N), `question` holds the **global message counter** (1..M) within the conv. `question_category` is null.
- `phase=ask` — one row per Q&A. `question_category` holds the LoCoMo **question category** (1=single-hop, 2=multi-hop, 3=temporal, 4=open-domain, 5=adversarial), `question` holds the prompt text. `seed` is null on ask rows; each question is asked exactly once. `judge_n` carries the LLM-as-judge panel size (default 3 calls), `judge_verdict` is the majority vote, and `judge_reason` carries the reasoning of the first judge call.

Timing (no rounding, all milliseconds):

| Column        | Meaning                                                                 |
| ------------- | ----------------------------------------------------------------------- |
| `wall_ms`     | `compute_ms + flush_ms` — total bench-measured time for the row.        |
| `compute_ms`  | `t_call_start` → HTTP response returned. **The user-facing latency a production system would pay.** |
| `flush_ms`    | Bench-side I/O quiescence wait we add *after* the response so async DB checkpoints (Neo4j) land in this row's `disk_cloud_bytes`. **Instrumentation artifact**, not a real cost. |

Resource counters (kernel-direct, no scrape interval):

| Column                    | Meaning                                                                 |
| ------------------------- | ----------------------------------------------------------------------- |
| `cpu_edge_ns`             | Cumulative CPU time consumed by edge containers during this call (ns). Edge = `coordinator`, `ollama`. |
| `cpu_cloud_ns`            | Same for cloud containers (`benchmark`, `memory`, `qdrant`, `neo4j`, `litellm`, `responder`, `toxiproxy`). |
| `ram_edge_peak_bytes`     | Per-call delta of edge group's `memory.peak` — additional working-set high-water mark induced by the call. Non-negative. |
| `ram_cloud_peak_bytes`    | Same for cloud group.                                                   |
| `disk_edge_bytes`         | Edge group's blkio read+write delta during the call (cgroup `io.stat`). |
| `disk_cloud_bytes`        | Same for cloud. Captures Neo4j async-checkpoint writes thanks to `flush_ms` quiescence wait. |
| `network_edge_bytes`      | TX-only bytes from edge containers (each byte counted once at sender; toxiproxy excluded). |
| `network_cloud_bytes`     | Same for cloud.                                                         |

LLM tokens (litellm `/metrics`, split by model name into edge vs cloud):

| Column               | Meaning                                                                 |
| -------------------- | ----------------------------------------------------------------------- |
| `edge_llm_tokens`    | Tokens served by ollama (model `qwen2.5:3b-instruct-q4_K_M`, free in litellm pricing). |
| `edge_llm_cost_usd`  | USD cost of edge LLM tokens (≈ 0).                                      |
| `cloud_llm_tokens`   | Tokens through OpenAI passthrough (`gpt-4o-mini`, `text-embedding-3-small`, ...). |
| `cloud_llm_cost_usd` | USD cost of cloud LLM tokens — the real bill.                           |
| `llm_tokens`         | `edge + cloud`.                                                         |
| `llm_cost_usd`       | `edge + cloud`.                                                         |

Resource counters are read directly from kernel cgroup pseudo-files and litellm's `/metrics` endpoint — every snapshot reflects the moment of the read, no TSDB middleman.


Responder context (per-`phase=ask` row, populated by responder→coordinator→benchmark plumbing):

| Column                       | Meaning                                                                 |
| ---------------------------- | ----------------------------------------------------------------------- |
| `responder_context_window_tokens`     | `prompt_tokens` of the OpenAI completion that produced the final answer — the actual context length the answering LLM consumed for this question. Used in §2b to compute `tokens_per_question` and `tokens_per_correct`. Null on warmup/load rows and on ask rows that errored before the final completion. |
| `responder_context_window_cost_usd`   | USD cost of the responder prompt slice, priced through litellm's cost table for the responder model. Already a slice of `cloud_llm_cost_usd` — do NOT add to LLM totals. |


LLM-as-judge consensus columns (per `phase=ask` row):

| Column                  | Meaning                                                                 |
| ----------------------- | ----------------------------------------------------------------------- |
| `judge_n`               | Number of independent judge calls per row, set by `LLM_AS_JUDGE_SEED` on the request (default 3). |
| `judge_verdict`         | Majority-vote final verdict (CORRECT iff strictly more than half the calls returned CORRECT, else WRONG; PLACEHOLDER/ERROR pass through if every call agrees). |
| `judge_reason`          | Reasoning string from the first judge call (only one is persisted; per-call labels are not written). |